- 예외설계

*우리 프로젝트에서 예외 부모*

In [7]:
class AgentError(Exception):
    status_code = 400
    cod = "agent_error"

    def __init__(self, message: str, *, detail: str|None = None):
        super().__init__(message) # print(e) -> 메서지 나옴
        self.message = message
        self.detail = detail

*자식 클래스로 세분화*

In [ ]:
# 요청한 자원이 없다
class NotFound(AgentError):
    status_code = 404
    code = "Not_found"

# 자원은 있으나, 이 사용자가 접근할 수 없다.
class PermissionDenied(AgentError):
    status_code = 403
    code = "Permission_denied"

# # 입력 가드에 걸림 - 길이 초과, 인젝션 의심 등
# class GuardTripped(AgentError):
#     status_code = 400
#     code = "guard_tripped"

# # 호출 한도를 넘었다.
# class RateLimited(AgentError):
#     status_code = 429
#     code = "rate_limited"

# # 승인 없이 외부 실행 시도
# class ApprovalRequired(AgentError):
#     status_code = 409
#     code "approval_required"

# # 모드 변경 불가 시
# class ModeNotAvailable(AgentError):
#     status_code = 409
#     code = "mode_not_available"

# # 입력값이 규칙에 맞지 앉는다
# class ValidationFailed(AgentError):
#     staus_code = 422
#     code = "Validation_failed"

# 외부 서비스 호출 실패
class ExternalServiceError(AgentError):
    status_code = 502
    code = "External_service_error"



e = NotFound("문서를 찾을 수 없습니다.", detail = "doc_id에 id가 없는 값")
print("메세지 : ", e)
print("상태코드 : ", e.status_code)
print("코드 : ", e.code)
print("상세 : ", e.detail)

메세지 :  문서를 찾을 수 없습니다.
상태코드 :  404
코드 :  Not_found
상세 :  doc_id에 id가 없는 값


*사용 예시*

In [11]:
def handle(exc):
    if isinstance(exc, AgentError):
        # 우리가 의도한 상황
        return {"status" : exc.status_code, "code" : exc.code, "message" : exc.message}

    return {"status" : 500, "code" : "internal_error", "message" : " 서버 오류가 발생했습니다."} # dict로 만들어서 json변환 해서 전송 가능

for exc in [
    NotFound("문서 못찾음"),
    PermissionDenied("이 문서를 볼 권한이 없습니다."),
    ExternalServiceError("문서 변환 서비스에 연결하지 못하였습니다."),
    KeyError("doc_id")
]:
    print(handle(exc))



{'status': 404, 'code': 'Not_found', 'message': '문서 못찾음'}
{'status': 403, 'code': 'Permission_denied', 'message': '이 문서를 볼 권한이 없습니다.'}
{'status': 502, 'code': 'External_service_error', 'message': '문서 변환 서비스에 연결하지 못하였습니다.'}
{'status': 500, 'code': 'internal_error', 'message': ' 서버 오류가 발생했습니다.'}


- raise... from...

In [12]:
import json 

def parse_bad(text):
    try:
        return json.loads(text)  # 문자열 -> 파이썬 객체 
    except json.JSONDecodeError as e:
        # 우리가 설계한 예외 클래스로 예외 발생시키기 
        raise ValidationFailed(
            "데이터 형식이 올바르지 않습니다.",  # 우리가 예외 메세지 추가 
            detail=str(e)   # 원래 터진 에러의 메세지 추가 (개발자용)
        ) from e  # 원인 연결 

try:
    parse_bad("{이건 json 아니야}")
except ValidationFailed as e:
    print("예외 메세지 : ", e)
    print("원인 : ", type(e.__cause__).__name__, " : ",e.__cause__, )




예외 메세지 :  데이터 형식이 올바르지 않습니다.
원인 :  JSONDecodeError  :  Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
